In [ ]:
# CELL 1: INSTALASI DAN IMPORT DEPENDENSI
!pip install pandas numpy scikit-learn haversine sentence-transformers
import pandas as pd
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity
from haversine import haversine, Unit
from sentence_transformers import SentenceTransformer
import json
import ast


In [ ]:
# CELL 2: LOAD & CLEANING DATA
# Load data
df = pd.read_csv('data.csv')

# Hapus duplikat berdasarkan kolom 'title'
df = df.drop_duplicates(subset=['title'])

# Hapus baris yang memiliki nilai kosong pada kolom 'latitude' atau 'longitude'
df = df.dropna(subset=['latitude', 'longitude'])

# Pastikan tipe data 'latitude' dan 'longitude' adalah float
df['latitude'] = df['latitude'].astype(float)
df['longitude'] = df['longitude'].astype(float)

# Isi nilai kosong di kolom 'title', 'category', dan 'review_rating'
df['title'] = df['title'].fillna('')
df['category'] = df['category'].fillna('')
df['review_rating'] = df['review_rating'].fillna('N/A')

df = df.reset_index(drop=True)


In [ ]:
# CELL 3: EKSTRAKSI JSON (SUPER KRUSIAL)
def parse_reviews(row_str):
    if pd.isna(row_str):
        return ""
    try:
        try:
            data = json.loads(row_str)
        except:
            data = ast.literal_eval(row_str)
        
        if isinstance(data, list):
            descriptions = []
            for item in data:
                if isinstance(item, dict) and 'Description' in item:
                    descriptions.append(str(item['Description']))
            return " ".join(descriptions)
        return ""
    except:
        return ""

def parse_about(row_str):
    if pd.isna(row_str):
        return ""
    try:
        try:
            data = json.loads(row_str)
        except:
            data = ast.literal_eval(row_str)
            
        if isinstance(data, list):
            names = []
            for item in data:
                if isinstance(item, dict) and 'options' in item and isinstance(item['options'], list):
                    for option in item['options']:
                        if isinstance(option, dict) and option.get('enabled') == True and 'name' in option:
                            names.append(str(option['name']))
            return ", ".join(names)
        return ""
    except:
        return ""

df['clean_reviews'] = df['user_reviews'].apply(parse_reviews)
df['clean_about'] = df['about'].apply(parse_about)

df['nlp_content'] = df['title'] + " | Kategori: " + df['category'] + " | Fasilitas: " + df['clean_about'] + " | Ulasan: " + df['clean_reviews']


In [ ]:
# CELL 4: EMBEDDING DENGAN NLP TRANSFORMERS
model = SentenceTransformer('paraphrase-multilingual-MiniLM-L12-v2')

print("Proses encoding teks menjadi embeddings (ini mungkin memakan waktu)...")
embeddings = model.encode(df['nlp_content'].tolist(), show_progress_bar=True)
df['embeddings'] = list(embeddings)
print("Selesai!")


In [ ]:
# CELL 5: HYBRID RECOMMENDER FUNCTION
def get_recommendations(user_query, user_lat, user_lon, max_distance_km=20, top_n=5):
    # 1. Encode user_query
    query_vec = model.encode([user_query])
    
    # 2. Hitung cosine similarity
    all_embeddings = np.vstack(df['embeddings'].values)
    sim_scores = cosine_similarity(query_vec, all_embeddings)[0]
    
    recommendations = []
    
    # 3. Iterasi ke setiap baris df
    user_loc = (user_lat, user_lon)
    for i in range(len(df)):
        row = df.iloc[i]
        place_loc = (row['latitude'], row['longitude'])
        distance = haversine(user_loc, place_loc, unit=Unit.KILOMETERS)
        
        # 4. Filter jarak
        if distance <= max_distance_km:
            recommendations.append({
                'Title': row['title'],
                'Category': row['category'],
                'Rating': row['review_rating'],
                'Jarak (Km)': round(distance, 2),
                'Skor NLP': sim_scores[i],
                'Cuplikan Ulasan': row['clean_reviews'][:150]
            })
            
    # 5. Konversi list ke dataframe
    res_df = pd.DataFrame(recommendations)
    if res_df.empty:
        print("Tidak ada tempat wisata yang ditemukan dalam radius tersebut.")
        return res_df
        
    # Urutkan descending berdasarkan Skor NLP dan return top_n
    res_df = res_df.sort_values(by='Skor NLP', ascending=False).head(top_n)
    res_df = res_df.drop(columns=['Skor NLP']).reset_index(drop=True)
    
    return res_df


In [ ]:
# CELL 6: DEMO PENGUJIAN SISTEM
user_lat = -7.250445
user_lon = 112.768845
user_query = "tempat yang suasananya sejuk, asri, banyak pepohonan rindang dan parkirannya luas"

print(f"Mencari rekomendasi untuk query: '{user_query}'")
print(f"Lokasi User: Surabaya ({user_lat}, {user_lon})")
print("-" * 50)

hasil_rekomendasi = get_recommendations(user_query, user_lat, user_lon, max_distance_km=20, top_n=5)
display(hasil_rekomendasi)


In [ ]:
# === AUTO-GENERATED THUMBNAIL EXPORT ===
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import os

# Find all existing figure axes
figures = [plt.figure(i) for i in plt.get_fignums()]
if figures:
    fig = figures[-1]  # Use last figure
else:
    # Create a summary figure from available data
    fig, ax = plt.subplots(figsize=(12, 6))
    ax.text(0.5, 0.5, 'Project Visualization', fontsize=24, 
            ha='center', va='center', transform=ax.transAxes,
            fontweight='bold', color='#2d3436')
    ax.set_facecolor('#f8f9fa')
    fig.patch.set_facecolor('#f8f9fa')
    ax.axis('off')

thumbnail_path = os.path.join(os.path.dirname(os.path.abspath('__file__')), 'thumbnail.png')
fig.savefig(thumbnail_path, dpi=150, bbox_inches='tight', 
            facecolor='white', edgecolor='none')
print(f"✅ Thumbnail saved: {thumbnail_path}")
plt.show()
